In [127]:
# import record into student list array to use
import csv

DEFAULT_TEAM_SIZE = 5

student_list = []

with open("records.csv", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        student = {
            "tutorial": row["Tutorial Group"].strip(),
            "student_id": row["Student ID"].strip(),
            "name": row["Name"].strip(),
            "school": row["School"].strip(),
            "gender": row["Gender"].strip(),
            "cgpa": float(row["CGPA"]),
        }
        student_list.append(student)

In [ ]:
tutorial_grp = {}
sorted_tutorial_grp = {}

for student in student_list:
    tutorial = student["tutorial"]
    if tutorial not in tutorial_grp:
        tutorial_grp[tutorial] = []
    tutorial_grp[tutorial].append(student)


{'G-1': [{'tutorial': 'G-1', 'student_id': '5002', 'name': 'Aarav Singh', 'school': 'CCDS', 'gender': 'Male', 'cgpa': 4.02}, {'tutorial': 'G-1', 'student_id': '3838', 'name': 'Aarti Nair', 'school': 'EEE', 'gender': 'Female', 'cgpa': 4.05}, {'tutorial': 'G-1', 'student_id': '2091', 'name': 'Adlan Bin Rahman', 'school': 'EEE', 'gender': 'Male', 'cgpa': 4.2}, {'tutorial': 'G-1', 'student_id': '288', 'name': 'Ajay Verma', 'school': 'CoB (NBS)', 'gender': 'Male', 'cgpa': 4.01}, {'tutorial': 'G-1', 'student_id': '4479', 'name': 'Amelia Kim', 'school': 'CCDS', 'gender': 'Female', 'cgpa': 4.11}, {'tutorial': 'G-1', 'student_id': '5708', 'name': 'Ananya Ramesh', 'school': 'SoH', 'gender': 'Male', 'cgpa': 4.2}, {'tutorial': 'G-1', 'student_id': '4563', 'name': 'Anjali Patel', 'school': 'WKW SCI', 'gender': 'Female', 'cgpa': 4.01}, {'tutorial': 'G-1', 'student_id': '3989', 'name': 'Anthony Liu', 'school': 'WKW SCI', 'gender': 'Male', 'cgpa': 4.15}, {'tutorial': 'G-1', 'student_id': '2115', 'name

In [129]:
# Create helpers


def max_allowed_per_attribute(team_size):
    return (team_size + 1) // 2


def counts_in_team(team, key):
    # dictionary to accumulate counts per attribute value
    counts_dict = {}
    for student in team:
        attribute_value = student[key]
        counts_dict[attribute_value] = counts_dict.get(attribute_value, 0) + 1
    return counts_dict


def violates_cap_after_add(team, student, key, team_size):
    max_allowed = max_allowed_per_attribute(team_size)
    current_counts = counts_in_team(team, key)
    attribute_value = student[key]
    return current_counts.get(attribute_value, 0) + 1 > max_allowed


def concentration_after_add(team, student, key):
    current_counts = counts_in_team(team, key)
    attribute_value = student[key]
    return current_counts.get(attribute_value, 0) + 1

In [130]:
# ==============================
# Placement score: diversity-first, then CGPA balance
# ==============================
def placement_score(team, student, tutorial_mean_t, team_size):
    # enum values
    HARD = 10_000  # block if it creates a cap violation
    SOFT_SCHOOL = 5  # gentle penalty for concentration
    SOFT_GENDER = 3
    SOFT_FILL = 1  # slightly prefer smaller teams now

    # Hard caps: do not place if it would exceed allowed count
    if violates_cap_after_add(team, student, "school", team_size):
        return HARD
    if violates_cap_after_add(team, student, "gender", team_size):
        return HARD

    # Soft penalties: encourage spreading out attributes
    school_conc = concentration_after_add(team, student, "school")
    gender_conc = concentration_after_add(team, student, "gender")

    # CGPA balance (via t_score): keep team avg close to tutorial mean
    new_sum = sum(m["t_score"] for m in team) + student["t_score"]
    new_sz = len(team) + 1
    cgpa_penalty = abs(new_sum / new_sz - tutorial_mean_t)

    # Fill penalty: slightly prefer teams with fewer members
    fill_penalty = len(team) * SOFT_FILL

    return (
        (SOFT_SCHOOL * school_conc)
        + (SOFT_GENDER * gender_conc)
        + cgpa_penalty
        + fill_penalty
    )

In [131]:
# %%
# ==============================
# Diversity-aware team builder
# ==============================
def build_teams(tutorial_id, students, team_size=DEFAULT_TEAM_SIZE):
    # Order students in "snake" to mix highs & lows
    students_sorted = sorted(students, key=lambda x: x["t_score"])
    snake = []
    L, R = 0, len(students_sorted) - 1
    while L <= R:
        snake.append(students_sorted[R])
        R -= 1
        if L <= R:
            snake.append(students_sorted[L])
            L += 1
    # means to sort them by [highest, lowest, 2nd highest, 2nd lowest, etc...]

    # Make empty teams
    num_teams = (len(students) + team_size - 1) // team_size
    teams = [[] for _ in range(num_teams)]

    tutorial_mean_t = sum(s["t_score"] for s in students) / len(students)

    # Greedy placement using placement_score
    for student in snake:
        best_team_index = None
        best_placement_score = float("inf")
        for team_index, current_team in enumerate(teams):
            if len(current_team) >= team_size:
                continue
            current_placement_score = placement_score(
                current_team, student, tutorial_mean_t, team_size
            )
            if current_placement_score < best_placement_score:
                best_placement_score, best_team_index = (
                    current_placement_score,
                    team_index,
                )
        # Fallback if all are full (shouldn't happen normally)
        if best_team_index is None:
            best_team_index = min(range(len(teams)), key=lambda k: len(teams[k]))
        teams[best_team_index].append(student)

    # Label teams
    labeled = []
    for i, team in enumerate(teams, 1):
        team_id = f"{tutorial_id}-T{i}"
        team_mean = sum(x["t_score"] for x in team) / len(team)
        for x in team:
            x["team_id"] = team_id
            x["team_tscore"] = round(team_mean, 3)
        labeled.append(team)

    return labeled

In [132]:
def repair_teams(teams, team_size, tutorial_mean_t, max_iterations=200, verbose=True):
    """
    Targeted repair:
      - Repeatedly find the two teams whose mean T-scores are farthest ABOVE and BELOW
        the tutorial mean (the extremes).
      - Try all 1-for-1 swaps between those two teams.
      - Commit the single swap that most reduces the sum of deviations from the tutorial mean,
        without creating any school/gender cap violations.
      - Stop when no improving swap exists (or max_iterations reached).

    Teams are modified in-place and relabeled at the end.
    """

    # ---- helpers (use your existing helpers where possible) ----
    def team_mean_t(team):
        return (sum(s["t_score"] for s in team) / len(team)) if team else 0.0

    def deviation(team):
        return team_mean_t(team) - tutorial_mean_t

    def max_allowed():
        return max_allowed_per_attribute(team_size)

    def cap_ok(team):
        cap = max_allowed()
        for attr in ("school", "gender"):
            for cnt in counts_in_team(team, attr).values():
                if cnt > cap:
                    return False
        return True

    def relabel(teams_):
        for i, t in enumerate(teams_, 1):
            if not t:
                continue
            tutorial_id = t[0]["tutorial"]
            tid = f"{tutorial_id}-T{i}"
            mean_t = team_mean_t(t)
            for s in t:
                s["team_id"] = tid
                s["team_tscore"] = round(mean_t, 3)

    # ---- main loop: only work on the two extreme teams each pass ----
    for _ in range(max_iterations):
        # compute deviations
        devs = [(i, deviation(t)) for i, t in enumerate(teams) if t]
        if len(devs) < 2:
            break

        # pick extremes
        hi_idx, hi_dev = max(devs, key=lambda x: x[1])  # most above mean
        lo_idx, lo_dev = min(devs, key=lambda x: x[1])  # most below mean

        # If both sides are already very close, we're done
        if abs(hi_dev) <= 1e-9 and abs(lo_dev) <= 1e-9:
            break

        A = teams[hi_idx]
        B = teams[lo_idx]

        # Try all swaps A[i] <-> B[j], keep the single best strictly improving swap
        base_cost = abs(deviation(A)) + abs(deviation(B))
        best = None
        best_cost = base_cost

        for ia, sa in enumerate(A):
            for ib, sb in enumerate(B):
                # simulate swap (size preserved)
                A2 = A[:]
                B2 = B[:]
                A2[ia], B2[ib] = sb, sa

                # respect caps
                if not (cap_ok(A2) and cap_ok(B2)):
                    continue

                # objective: sum of absolute deviations from tutorial mean (for these two teams only)
                cost = abs(
                    (sum(s["t_score"] for s in A2) / len(A2)) - tutorial_mean_t
                ) + abs((sum(s["t_score"] for s in B2) / len(B2)) - tutorial_mean_t)

                if cost + 1e-12 < best_cost:
                    best_cost = cost
                    best = (ia, ib)

        if best is None:
            # no improving swap between extremes; stop
            if verbose:
                print("[repair] No improving swap between extreme teams; done.")
            break

        # commit the best swap
        ia, ib = best
        A[ia], B[ib] = B[ib], A[ia]
        if verbose:
            print(
                f"[repair] Swap improved cost {base_cost:.4f} -> {best_cost:.4f} | "
                f"Teams: {hi_idx+1} (high) <-> {lo_idx+1} (low)"
            )

    # relabel & return
    relabel(teams)
    return teams

In [133]:
# %%
# ==============================
# Main per-tutorial allocator (t-score + diversity assignment + repair)
# ==============================
def create_groups_by_tscore(tutorial_id, students, team_size=DEFAULT_TEAM_SIZE):
    """
    Compute t-scores; assign to diverse teams with CGPA balance; repair; print summary.
    Returns: list of teams (list[list[student]])
    """

    # --- Step 1: min/max CGPA for normalization ---
    cgpas = [s["cgpa"] for s in students]
    min_cgpa, max_cgpa = min(cgpas), max(cgpas)
    cgpa_range = max_cgpa - min_cgpa if max_cgpa != min_cgpa else 1

    # --- Step 2: counts in tutorial ---
    gender_counts = {}
    school_counts = {}

    for student in students:
        gender_counts[student["gender"]] = gender_counts.get(student["gender"], 0) + 1
        school_counts[student["school"]] = school_counts.get(student["school"], 0) + 1

    print(gender_counts, school_counts)
    min_gender_count = min(gender_counts.values()) if gender_counts else 0
    min_school_count = min(school_counts.values()) if school_counts else 0

    # --- Step 3: compute t-scores for each students ---
    total_tscore = 0
    for student in students:
        normalized_cgpa = (student["cgpa"] - min_cgpa) / cgpa_range
        gender_diversity_bonus = (
            0.2 if gender_counts[student["gender"]] == min_gender_count else 0.0
        )  # 0.2 bonus if gender is the least in a tutorial
        school_diversity_bonus = (
            0.1 if school_counts[student["school"]] == min_school_count else 0.0
        )  # +0.1 if school count is the least in a tutorial
        t_score = normalized_cgpa + gender_diversity_bonus + school_diversity_bonus

        student["t_score"] = round(t_score, 3)
        total_tscore += student["t_score"]

    # --- Step 4: tutorial mean T-score ---
    mean_tscore = total_tscore / len(students)

    # --- Step 5: build initial teams with diversity-aware placement ---
    teams = build_teams(tutorial_id, students, team_size)

    # --- Step 6: repair teams to reduce violations and polish CGPA balance ---
    # TODO: comment out if its hard to understand
    teams = repair_teams(teams, team_size, mean_tscore)

    # --- Step 7: summary printout ---
    team_mean_diffs = [
        abs(sum(student["t_score"] for student in t) / len(t) - mean_tscore)
        for t in teams
    ]
    avg_team_diff = sum(team_mean_diffs) / len(team_mean_diffs)

    print(f"\n=== {tutorial_id} summary ===")
    print(f"Tutorial mean T-score: {mean_tscore:.3f}")
    print(f"Actual average team diff from tutorial mean: {avg_team_diff:.4f}")

    for team in teams:
        team_mean = sum(s["t_score"] for s in team) / len(team)
        diff = abs(team_mean - mean_tscore)
        print(
            f"{tutorial_id} | {team[0]['team_id']} | Team mean T={team_mean:.3f} | Diff={diff:.3f}"
        )

    return teams

In [134]:
# run algorithm
for tutorial_id, students in tutorial_grp.items():
    teams = create_groups_by_tscore(tutorial_id, students, DEFAULT_TEAM_SIZE)
    sorted_tutorial_grp[tutorial_id] = teams

{'Male': 21, 'Female': 29} {'CCDS': 5, 'EEE': 10, 'CoB (NBS)': 8, 'SoH': 5, 'WKW SCI': 2, 'CoE': 3, 'MAE': 4, 'SPMS': 2, 'SBS': 1, 'SSS': 5, 'ASE': 1, 'NIE': 1, 'ADM': 1, 'CCEB': 1, 'MSE': 1}
[repair] Swap improved cost 0.1552 -> 0.0936 | Teams: 1 (high) <-> 10 (low)
[repair] Swap improved cost 0.1492 -> 0.0349 | Teams: 3 (high) <-> 2 (low)
[repair] Swap improved cost 0.1098 -> 0.0558 | Teams: 1 (high) <-> 4 (low)
[repair] Swap improved cost 0.0762 -> 0.0146 | Teams: 1 (high) <-> 5 (low)
[repair] Swap improved cost 0.0596 -> 0.0053 | Teams: 2 (high) <-> 10 (low)
[repair] Swap improved cost 0.0366 -> 0.0017 | Teams: 6 (high) <-> 8 (low)
[repair] Swap improved cost 0.0250 -> 0.0077 | Teams: 1 (high) <-> 4 (low)
[repair] Swap improved cost 0.0126 -> 0.0021 | Teams: 9 (high) <-> 4 (low)
[repair] Swap improved cost 0.0086 -> 0.0033 | Teams: 2 (high) <-> 5 (low)
[repair] Swap improved cost 0.0048 -> 0.0012 | Teams: 10 (high) <-> 2 (low)
[repair] No improving swap between extreme teams; done.

In [135]:
# export to csv
output_data = []
for tutorial_id, teams in sorted_tutorial_grp.items():
    for team in teams:
        for student in team:
            row = {
                "Tutorial Group": student["tutorial"],
                "Student ID": student["student_id"],
                "School": student["school"],
                "Name": student["name"],
                "Gender": student["gender"],
                "CGPA": student["cgpa"],
                "Team Assigned": student.get("team_id", f"{tutorial_id}-Unknown"),
            }
            output_data.append(row)

output_file = "./output.csv"
fieldnames = [
    "Tutorial Group",
    "Student ID",
    "School",
    "Name",
    "Gender",
    "CGPA",
    "Team Assigned",
]

with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(output_data)

print(f"\nTeam assignments exported to {output_file}")
print(f"Total students processed: {len(output_data)}")
print(
    f"Total teams created: {sum(len(teams) for teams in sorted_tutorial_grp.values())}"
)


Team assignments exported to ./output.csv
Total students processed: 6000
Total teams created: 1200
